# 07 Sequence models (Family C)
For payment *t* the input is the sender's previous 10 payments plus the current one. We compare GRU, LSTM and a Transformer encoder, alone and combined with tabular features.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
import json
from presage.data.loaders import load_sim, load_features
from presage.models.sequence import step_features, build_sequences, train_seq
from presage.models.graph_train import prep_tab
from presage.evaluation.metrics import temporal_split
from sklearn.metrics import average_precision_score
df, acc = load_sim(); X = load_features(); g = json.load(open(ROOT/'data/processed/sim_v1_feature_groups.json'))
cust = df.src_is_customer.values==1; tr0,va0,te0 = temporal_split(df.day.values); tr, va, te = tr0&cust, va0&cust, te0&cust; y = df.is_app_fraud.values.astype(np.float32)
steps = step_features(df, X); mu, sd = steps[tr].mean(0), steps[tr].std(0)+1e-6
seq, mask = build_sequences(df, (steps-mu)/sd, 10); print(seq.shape, 'positions filled (current payment last):', mask.mean(0).round(2))

(390498, 11, 11) positions filled (current payment last): [0.84 0.85 0.86 0.87 0.89 0.9  0.92 0.93 0.95 0.97 1.  ]


**Leak check by construction:** steps before the current one come only from earlier payments of the same sender (`build_sequences` sorts by sender then time).

In [3]:
import torch; torch.set_num_threads(2)
out = {}
for kind in ['gru', 'transformer']:
    m, pred = train_seq(kind, seq, mask, y, tr, va, epochs=3, verbose=False)
    out[kind] = average_precision_score(y[te], pred(np.where(te)[0]))
print('sequence-only test PR-AUC (3 epochs, demo):', out)

sequence-only test PR-AUC (3 epochs, demo): {'gru': 0.38851278769941305, 'transformer': 0.3295065076032057}


Sequence-only models see behaviour but not the beneficiary's network state, so they trail the tabular+graph models. Their value is as the behaviour stream inside PRESAGE (`z_beh`). Full-length comparison is in the PRESAGE ablation (`no_beh`).

In [4]:
a = table('presage_full_s42.csv'); a

,variant,pr_auc,pr_auc_lo,pr_auc_hi,roc_auc,recall,precision,f1,f2,mcc,fpr,cost_costthr,prevented_ratio,mule_pr_auc,train_s,latency_ms,n_params
0,full,0.91934,0.870328,0.951497,0.985794,0.858065,0.953405,0.903226,0.875576,0.904039,0.000206,8315.22628,0.974992,0.996176,439.078402,0.343934,229517
